# Hybrid Retrieval Evaluation on TREC DL 2019/2020

This notebook organizes the retrieval pipeline into clear sections:

1. **Imports and setup**
2. **Dataset loading and judged-query filtering**
3. **Document pool construction**
4. **BM25 retrieval**
5. **Dense retrieval**
6. **Evaluation helpers**
7. **Fusion sweep**
8. **Redundancy and irrelevance analysis**

The goal is to keep the logic the same as the original implementation, while making the notebook easier to read, reuse, and extend.

In [1]:
# === 1. Imports and setup ===
# Core libraries for datasets, retrieval, evaluation, and analysis.

import re
from collections import defaultdict
from itertools import combinations

import ir_datasets
import numpy as np
import pandas as pd
import pytrec_eval
import torch
from tqdm.auto import tqdm

from nltk.stem import PorterStemmer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Warm up ir_datasets / verify installation.
_ = ir_datasets.load("msmarco-passage/train").docs_count()

# Global text preprocessing tools for BM25 and lexical redundancy.
ps = PorterStemmer()
STOP = set(ENGLISH_STOP_WORDS)

def tok(text: str):
    """Tokenize, lowercase, remove stopwords, and apply Porter stemming."""
    tokens = re.findall(r"\b\w+\b", text.lower())
    tokens = [t for t in tokens if t not in STOP]
    tokens = [ps.stem(t) for t in tokens]
    return tokens

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


## 2. Load datasets and keep only judged queries

We use the TREC DL 2019 and 2020 judged subsets from MS MARCO Passage.  
Only queries that have qrels are kept, and qrels are stored in the standard nested format:

- `queries[qid] = query text`
- `qrels[qid][docid] = relevance grade`
- `qid_source[qid] = dataset year`

In [2]:
# === 2. Dataset loading and judged-query filtering ===

DS_2019 = "msmarco-passage/trec-dl-2019"
DS_2020 = "msmarco-passage/trec-dl-2020"

def load_judged_only(ds, source_tag):
    """Return only queries that have judged qrels for a given dataset split."""
    queries_all = {str(q.query_id): q.text for q in ds.queries_iter()}
    qrels = defaultdict(dict)

    for qr in ds.qrels_iter():
        qid = str(qr.query_id)
        docid = str(qr.doc_id)
        qrels[qid][docid] = int(qr.relevance)

    judged_qids = set(qrels.keys())
    queries = {qid: queries_all[qid] for qid in judged_qids}
    qid_source = {qid: source_tag for qid in judged_qids}
    return queries, qrels, qid_source

def add_split(global_queries, global_qrels, global_qid_source,
              split_queries, split_qrels, split_source):
    """Merge one dataset split into the combined structures.
    If the same qid appears with different text, append the source tag.
    """
    for qid, qtext in split_queries.items():
        new_qid = qid
        if new_qid in global_queries and global_queries[new_qid] != qtext:
            new_qid = f"{qid}_{split_source[qid]}"

        global_queries[new_qid] = qtext
        global_qid_source[new_qid] = split_source[qid]

        for docid, grade in split_qrels[qid].items():
            global_qrels[new_qid][docid] = int(grade)

def counts_by_threshold(qid, qrels):
    """Summarize grade distribution for one query."""
    grades = list(qrels[qid].values())
    return {
        "num_qrels": len(grades),
        ">=1": sum(g >= 1 for g in grades),
        ">=2": sum(g >= 2 for g in grades),
        ">=3": sum(g >= 3 for g in grades),
        "max_grade": max(grades) if grades else None,
    }

ds19 = ir_datasets.load(DS_2019)
ds20 = ir_datasets.load(DS_2020)

print("Loaded:", DS_2019, "| queries:", ds19.queries_count(), "qrels:", ds19.qrels_count())
print("Loaded:", DS_2020, "| queries:", ds20.queries_count(), "qrels:", ds20.qrels_count())

queries19, qrels19, src19 = load_judged_only(ds19, "2019")
queries20, qrels20, src20 = load_judged_only(ds20, "2020")

queries = {}
qrels = defaultdict(dict)
qid_source = {}

add_split(queries, qrels, qid_source, queries19, qrels19, src19)
add_split(queries, qrels, qid_source, queries20, qrels20, src20)

print("Combined judged queries:", len(queries))
print("Unique relevance grades:", sorted({g for qid in qrels for g in qrels[qid].values()}))

Loaded: msmarco-passage/trec-dl-2019 | queries: 200 qrels: 9260
Loaded: msmarco-passage/trec-dl-2020 | queries: 200 qrels: 11386
Combined judged queries: 97
Unique relevance grades: [0, 1, 2, 3]


In [3]:
# Quick dataset sanity checks.

year_counts = pd.Series(list(qid_source.values())).value_counts().sort_index()
display(year_counts.to_frame("num_queries"))

sample_qids = list(queries.keys())[:10]
df_counts = pd.DataFrame([
    {"qid": qid, "year": qid_source[qid], **counts_by_threshold(qid, qrels)}
    for qid in sample_qids
])
display(df_counts)

example_qid = list(queries.keys())[0]
top_judged = sorted(qrels[example_qid].items(), key=lambda x: x[1], reverse=True)[:10]

print("Example qid:", example_qid, "| year:", qid_source[example_qid])
print("Query:", queries[example_qid])
print("Num judged docs:", len(qrels[example_qid]))
print("Top judged (docid, grade):")
for docid, grade in top_judged:
    print("  ", docid, grade)

,num_queries
2019,43
2020,54


,qid,year,num_qrels,>=1,>=2,>=3,max_grade
0,915593,2019,192,92,79,30,3
1,451602,2019,220,154,100,65,3
2,1124210,2019,330,139,120,6,3
3,130510,2019,133,28,14,6,3
4,87181,2019,158,83,31,0,2
5,1115776,2019,152,24,4,2,3
6,1113437,2019,180,77,25,2,3
7,1121709,2019,178,12,3,0,2
8,207786,2019,137,24,11,0,2
9,962179,2019,161,25,21,20,3


Example qid: 915593 | year: 2019
Query: what types of food can you cook sous vide
Num judged docs: 192
Top judged (docid, grade):
   1605506 3
   210384 3
   2588143 3
   275728 3
   2923493 3
   2923498 3
   3385968 3
   3538160 3
   3538168 3
   3573478 3


## 3. Build the graded candidate document pool

Instead of searching the full MS MARCO corpus, this notebook evaluates on the union of all judged documents across TREC DL 2019 and 2020.  
This creates a much smaller candidate set while preserving the graded labels used for evaluation.

In [4]:
# === 3. Build candidate pool and fetch document text ===

candidate_docids = sorted({docid for qid in qrels for docid in qrels[qid]})
print("Candidate docids (union of judged docs):", len(candidate_docids))

docstore = ds19.docs_store()  # same underlying corpus for these splits
doc_text_map = {}
missing = []

for did in tqdm(candidate_docids, desc="Fetching docs by id"):
    try:
        doc_text_map[did] = docstore.get(did).text
    except KeyError:
        missing.append(did)

print("Fetched docs:", len(doc_text_map))
print("Missing docs:", len(missing))

docids = list(doc_text_map.keys())
docs = [doc_text_map[did] for did in docids]

Candidate docids (union of judged docs): 20349


Fetching docs by id:   0%|          | 0/20349 [00:00<?, ?it/s]

Fetched docs: 20349
Missing docs: 0


## 4. BM25 retrieval baseline

This section tokenizes the candidate corpus, fits a BM25 model, retrieves the top documents for each query, and evaluates with nDCG@10.

In [5]:
# === 4. BM25 retrieval ===

K_RETRIEVE = 100

corpus_tokens = [tok(doc_text_map[did]) for did in docids]
bm25 = BM25Okapi(corpus_tokens)

run_bm25 = {}
bm25_scores_cache = {}

for qid, qtext in tqdm(queries.items(), desc="BM25 scoring"):
    scores = bm25.get_scores(tok(qtext))
    bm25_scores_cache[qid] = scores

    topk = np.argpartition(scores, -K_RETRIEVE)[-K_RETRIEVE:]
    topk = topk[np.argsort(scores[topk])[::-1]]
    run_bm25[qid] = {docids[i]: float(scores[i]) for i in topk}

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {"ndcg_cut_10"})
results_bm25 = evaluator.evaluate(run_bm25)
avg_ndcg10_bm25 = np.mean([r["ndcg_cut_10"] for r in results_bm25.values()])

print(f"BM25 avg nDCG@10: {avg_ndcg10_bm25:.4f}")

BM25 scoring:   0%|          | 0/97 [00:00<?, ?it/s]

BM25 avg nDCG@10: 0.4610


## 5. Dense retrieval baseline

This section encodes all candidate documents and queries using `all-MiniLM-L6-v2`, then ranks documents by cosine similarity.

Because embeddings are normalized, cosine similarity is equivalent to the dot product.

In [6]:
# === 5. Dense retrieval ===

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

doc_emb = model.encode(
    docs,
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

qids = list(queries.keys())
qtexts = [queries[qid] for qid in qids]

query_emb = model.encode(
    qtexts,
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

run_dense = {}

for qi, qid in enumerate(tqdm(qids, desc="Dense retrieval")):
    scores = doc_emb @ query_emb[qi]
    topk = np.argpartition(scores, -K_RETRIEVE)[-K_RETRIEVE:]
    topk = topk[np.argsort(scores[topk])[::-1]]
    run_dense[qid] = {docids[i]: float(scores[i]) for i in topk}

results_dense = evaluator.evaluate(run_dense)
avg_ndcg10_dense = np.mean([r["ndcg_cut_10"] for r in results_dense.values()])

print(f"Dense avg nDCG@10: {avg_ndcg10_dense:.4f}")

Batches:   0%|          | 0/159 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Dense retrieval:   0%|          | 0/97 [00:00<?, ?it/s]

Dense avg nDCG@10: 0.6750


## 6. Evaluation helper functions

These helpers support three kinds of analysis:

- **Fusion** of sparse and dense scores
- **Irrelevance**: how many top-k documents have grade 0
- **Redundancy**: average pairwise similarity among top-k retrieved documents

In [7]:
# === 6. Evaluation helpers ===

def topk_docids_from_run(run_any, qid, k=10):
    """Return the top-k docids for one query from a run dict."""
    items = run_any[str(qid)]
    return [d for d, _ in sorted(items.items(), key=lambda x: x[1], reverse=True)[:k]]

def irrelevance_count_for_query(run_any, qid, qrels, k=10):
    """Count top-k retrieved docs with grade 0. Missing qrels are treated as 0."""
    judged = qrels.get(str(qid), {})
    top_docids = topk_docids_from_run(run_any, qid, k=k)
    return sum(int(judged.get(str(d), 0)) == 0 for d in top_docids)

def irrelevance_summary(run_any, qrels, queries, k=10):
    """Return average number/rate of irrelevant docs in top-k."""
    per_query = {
        str(qid): irrelevance_count_for_query(run_any, qid, qrels, k=k)
        for qid in queries
    }
    avg_zero_count = float(np.mean(list(per_query.values())))
    avg_zero_rate = avg_zero_count / k
    return avg_zero_count, avg_zero_rate, per_query

def minmax_norm(scores_dict):
    """Min-max normalize a {docid: score} dictionary."""
    if not scores_dict:
        return {}
    vals = np.fromiter(scores_dict.values(), dtype=float)
    vmin, vmax = float(vals.min()), float(vals.max())
    if vmax == vmin:
        return {k: 1.0 for k in scores_dict}
    return {k: (float(v) - vmin) / (vmax - vmin) for k, v in scores_dict.items()}

def fuse_runs_minmax(run_dense, run_sparse, alpha, qids=None, k=100):
    """Fuse dense and sparse runs with min-max normalization.
    alpha weights dense scores; (1-alpha) weights sparse scores.
    """
    if qids is None:
        qids = run_dense.keys()

    beta = 1.0 - alpha
    run_fused = {}

    for qid in qids:
        dense_norm = minmax_norm(run_dense.get(qid, {}))
        sparse_norm = minmax_norm(run_sparse.get(qid, {}))

        combined = {
            d: alpha * dense_norm.get(d, 0.0) + beta * sparse_norm.get(d, 0.0)
            for d in (set(dense_norm) | set(sparse_norm))
        }

        top = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:k]
        run_fused[str(qid)] = {str(docid): float(score) for docid, score in top}

    return run_fused

docid_to_i = {docid: i for i, docid in enumerate(docids)}

def mean_pairwise_cosine_from_docids(top_docids, doc_emb, docid_to_i):
    """Mean pairwise cosine similarity among retrieved docs."""
    idxs = [docid_to_i[d] for d in top_docids if d in docid_to_i]
    if len(idxs) < 2:
        return 0.0
    E = doc_emb[idxs]
    S = E @ E.T
    iu = np.triu_indices(len(idxs), k=1)
    return float(S[iu].mean())

def max_pairwise_cosine_from_docids(top_docids, doc_emb, docid_to_i):
    """Maximum pairwise cosine similarity among retrieved docs."""
    idxs = [docid_to_i[d] for d in top_docids if d in docid_to_i]
    if len(idxs) < 2:
        return 0.0
    E = doc_emb[idxs]
    S = E @ E.T
    iu = np.triu_indices(len(idxs), k=1)
    return float(S[iu].max())

def avg_redundancy_for_run(run_any, doc_emb, docid_to_i, qids=None, k_red=10):
    """Average mean pairwise cosine similarity over queries."""
    if qids is None:
        qids = run_any.keys()
    vals = []
    for qid in qids:
        top_docids = topk_docids_from_run(run_any, qid, k=k_red)
        vals.append(mean_pairwise_cosine_from_docids(top_docids, doc_emb, docid_to_i))
    return float(np.mean(vals))

## 7. Fusion sweep

We interpolate min-max normalized dense and BM25 scores:

\[
\text{fused}(d) = \alpha \cdot \text{dense}(d) + (1-\alpha) \cdot \text{bm25}(d)
\]

For each value of `alpha`, we report:

- average nDCG@10
- semantic redundancy@10
- irrelevance@10

In [8]:
# === 7. Fusion sweep ===

K_RED = 10
K_IRREL = 10
fusion_rows = []

for a in range(0, 101, 5):
    alpha = a / 100.0
    run_fused = fuse_runs_minmax(run_dense, run_bm25, alpha, qids=queries.keys(), k=K_RETRIEVE)

    res = evaluator.evaluate(run_fused)
    avg_ndcg = float(np.mean([r["ndcg_cut_10"] for r in res.values()]))
    avg_red = avg_redundancy_for_run(run_fused, doc_emb, docid_to_i, qids=queries.keys(), k_red=K_RED)
    avg0, rate0, _ = irrelevance_summary(run_fused, qrels, queries, k=K_IRREL)

    fusion_rows.append({
        "alpha_dense": alpha,
        "beta_bm25": 1.0 - alpha,
        "ndcg@10": avg_ndcg,
        f"redundancy@{K_RED}": avg_red,
        f"irrelevance@{K_IRREL}_avg_count": avg0,
        f"irrelevance@{K_IRREL}_rate": rate0,
    })

fusion_df = pd.DataFrame(fusion_rows)
display(fusion_df)

best_row = fusion_df.loc[fusion_df["ndcg@10"].idxmax()]
print(
    f"Best fusion by nDCG@10 -> alpha={best_row['alpha_dense']:.2f}, "
    f"beta={best_row['beta_bm25']:.2f}, ndcg@10={best_row['ndcg@10']:.4f}"
)

,alpha_dense,beta_bm25,ndcg@10,redundancy@10,irrelevance@10_avg_count,irrelevance@10_rate
0,0.00,1.00,0.461049,0.547228,4.463918,0.446392
1,0.05,0.95,0.469615,0.556565,4.391753,0.439175
2,0.10,0.90,0.486301,0.568741,4.195876,0.419588
3,0.15,0.85,0.507500,0.585441,4.000000,0.400000
4,0.20,0.80,0.525492,0.596953,3.876289,0.387629
5,0.25,0.75,0.535782,0.607565,3.773196,0.377320
6,0.30,0.70,0.556645,0.623698,3.556701,0.355670
7,0.35,0.65,0.574312,0.640981,3.298969,0.329897
8,0.40,0.60,0.593464,0.653383,3.144330,0.314433
9,0.45,0.55,0.611575,0.669121,3.010309,0.301031


Best fusion by nDCG@10 -> alpha=0.90, beta=0.10, ndcg@10=0.6772


## 8. Redundancy and irrelevance diagnostics

Finally, we compare BM25 and dense retrieval directly on:

- **Semantic redundancy** using embedding cosine similarity
- **Lexical redundancy** using Jaccard similarity over token sets
- **Irrelevance** using the number of grade-0 documents in the top-10

In [9]:
# === 8. Baseline redundancy and irrelevance diagnostics ===

doc_token_sets = {did: set(tok(doc_text_map[did])) for did in docids}

def avg_pairwise_jaccard(docids_topk, doc_token_sets):
    """Mean lexical Jaccard similarity across all unordered doc pairs."""
    pairs = list(combinations(docids_topk, 2))
    if not pairs:
        return 0.0
    sims = []
    for a, b in pairs:
        A, B = doc_token_sets[a], doc_token_sets[b]
        denom = len(A | B)
        sims.append(0.0 if denom == 0 else len(A & B) / denom)
    return float(np.mean(sims))

avg_redundancy_bm25 = avg_redundancy_for_run(run_bm25, doc_emb, docid_to_i, qids=queries.keys(), k_red=10)
avg_redundancy_dense = avg_redundancy_for_run(run_dense, doc_emb, docid_to_i, qids=queries.keys(), k_red=10)

redundancy_jacc_bm25 = {
    qid: avg_pairwise_jaccard(topk_docids_from_run(run_bm25, qid, k=10), doc_token_sets)
    for qid in queries.keys()
}
avg_redundancy_jacc_bm25 = float(np.mean(list(redundancy_jacc_bm25.values())))

max_redund_dense = {
    qid: max_pairwise_cosine_from_docids(topk_docids_from_run(run_dense, qid, k=10), doc_emb, docid_to_i)
    for qid in queries.keys()
}
avg_max_redund_dense = float(np.mean(list(max_redund_dense.values())))

avg0_bm25, rate0_bm25, _ = irrelevance_summary(run_bm25, qrels, queries, k=10)
avg0_dense, rate0_dense, _ = irrelevance_summary(run_dense, qrels, queries, k=10)

summary_df = pd.DataFrame([
    {
        "run": "BM25",
        "ndcg@10": avg_ndcg10_bm25,
        "semantic_redundancy@10": avg_redundancy_bm25,
        "lexical_redundancy@10": avg_redundancy_jacc_bm25,
        "irrelevance@10_avg_count": avg0_bm25,
        "irrelevance@10_rate": rate0_bm25,
    },
    {
        "run": "Dense",
        "ndcg@10": avg_ndcg10_dense,
        "semantic_redundancy@10": avg_redundancy_dense,
        "max_pairwise_cosine@10": avg_max_redund_dense,
        "irrelevance@10_avg_count": avg0_dense,
        "irrelevance@10_rate": rate0_dense,
    },
])

display(summary_df)

,run,ndcg@10,semantic_redundancy@10,lexical_redundancy@10,irrelevance@10_avg_count,irrelevance@10_rate,max_pairwise_cosine@10
0,BM25,0.461049,0.547437,0.151974,4.474227,0.447423,NaN
1,Dense,0.674993,0.735871,NaN,2.226804,0.222680,0.9632


## Notes

- This notebook evaluates retrieval over the **union of judged documents**, not the full MS MARCO corpus.
- That makes the experiment much cheaper, but the scores are not directly comparable to full-corpus leaderboard numbers.
- The current setup already makes it easy to plug in new fusion methods or reranking strategies by producing another `run` dictionary.
- If you want, the next natural cleanup step is to wrap the entire workflow into a few reusable functions or a small experiment class.

In [28]:
# === 9. Build adversarial query set for all judged queries ===
# For each judged query, create 50 perturbed variants by appending unrelated
# semantic content. Since these synthetic queries do not have official qrels,
# we reuse the original query's qrels as proxy labels.

adversarial_suffixes = [
    "my uncle is really good at basketball and loves lebron james",
    "i have been watching a lot of nba games recently",
    "the milky way galaxy contains billions of stars",
    "my dog just learned how to sit and roll over",
    "i am studying quantum mechanics this semester",
    "bitcoin prices have been extremely volatile lately",
    "my favorite soccer team just won the championship",
    "airplanes fly thousands of miles across oceans",
    "neural networks are used in modern ai systems",
    "my friend is training for a marathon this year",
    "the stock market dropped sharply yesterday",
    "my sister is learning how to play the violin",
    "volcanoes erupt when magma reaches the surface",
    "climate change affects weather patterns globally",
    "the new iphone has an upgraded camera system",
    "astronauts live on the international space station",
    "i just started reading a novel about dragons",
    "my car needs new tires before winter starts",
    "penguins live in very cold environments",
    "my brother is studying architecture in college",
    "electric cars are becoming more popular",
    "ancient egypt built large pyramids",
    "my friend just adopted a small kitten",
    "the olympics include many different sports",
    "i recently learned how to code in python",
    "sharks are powerful predators in the ocean",
    "classical music composers wrote symphonies",
    "my cousin collects vintage baseball cards",
    "mount everest is the tallest mountain",
    "satellites orbit the earth every day",
    "the world cup happens every four years",
    "i like hiking in national parks",
    "my laptop battery keeps dying quickly",
    "dolphins communicate using sound signals",
    "my friend is building a gaming pc",
    "the amazon rainforest has huge biodiversity",
    "my neighbor just bought a new motorcycle",
    "chess is a game of strategy and planning",
    "hurricanes form over warm ocean water",
    "my roommate watches a lot of anime",
    "people train for triathlons every year",
    "mars rovers explore the surface of mars",
    "my professor studies machine learning models",
    "dogs have an excellent sense of smell",
    "my friend plays guitar in a rock band",
    "glaciers slowly move across landscapes",
    "the super bowl is a huge sporting event",
    "my cousin is learning japanese",
    "whales migrate thousands of miles",
    "my teacher loves talking about world history",
]

queries_adv_all = {}
qrels_adv_all = defaultdict(dict)
qid_source_adv_all = {}
adv_to_source_qid = {}

for qid, qtext in queries.items():
    for i, suffix in enumerate(adversarial_suffixes, start=1):
        adv_qid = f"{qid}_adv_{i:02d}"
        adv_query = f"{qtext}. {suffix}"

        queries_adv_all[adv_qid] = adv_query
        qid_source_adv_all[adv_qid] = qid_source[qid]
        adv_to_source_qid[adv_qid] = qid

        # proxy labels: reuse original query qrels
        for docid, grade in qrels[qid].items():
            qrels_adv_all[adv_qid][docid] = int(grade)

print("Original judged queries:", len(queries))
print("Adversarial queries:", len(queries_adv_all))
print("Suffixes per query:", len(adversarial_suffixes))

Original judged queries: 97
Adversarial queries: 4850
Suffixes per query: 50


In [29]:
# === 10. BM25 retrieval on adversarial query set ===

run_bm25_adv = {}
bm25_scores_cache_adv = {}

for qid, qtext in tqdm(queries_adv_all.items(), desc="BM25 scoring (adversarial set)"):
    scores = bm25.get_scores(tok(qtext))
    bm25_scores_cache_adv[qid] = scores

    topk = np.argpartition(scores, -K_RETRIEVE)[-K_RETRIEVE:]
    topk = topk[np.argsort(scores[topk])[::-1]]
    run_bm25_adv[qid] = {docids[i]: float(scores[i]) for i in topk}

evaluator_adv = pytrec_eval.RelevanceEvaluator(qrels_adv_all, {"ndcg_cut_10"})
results_bm25_adv = evaluator_adv.evaluate(run_bm25_adv)
avg_ndcg10_bm25_adv = np.mean([r["ndcg_cut_10"] for r in results_bm25_adv.values()])

print(f"BM25 adversarial avg nDCG@10: {avg_ndcg10_bm25_adv:.4f}")

BM25 scoring (adversarial set):   0%|          | 0/4850 [00:00<?, ?it/s]

BM25 adversarial avg nDCG@10: 0.3676


In [30]:
# === 11. Dense retrieval on adversarial query set ===

adv_qids = list(queries_adv_all.keys())
adv_qtexts = [queries_adv_all[qid] for qid in adv_qids]

query_emb_adv = model.encode(
    adv_qtexts,
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

run_dense_adv = {}

for qi, qid in enumerate(tqdm(adv_qids, desc="Dense retrieval (adversarial set)")):
    scores = doc_emb @ query_emb_adv[qi]
    topk = np.argpartition(scores, -K_RETRIEVE)[-K_RETRIEVE:]
    topk = topk[np.argsort(scores[topk])[::-1]]
    run_dense_adv[qid] = {docids[i]: float(scores[i]) for i in topk}

results_dense_adv = evaluator_adv.evaluate(run_dense_adv)
avg_ndcg10_dense_adv = np.mean([r["ndcg_cut_10"] for r in results_dense_adv.values()])

print(f"Dense adversarial avg nDCG@10: {avg_ndcg10_dense_adv:.4f}")

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Dense retrieval (adversarial set):   0%|          | 0/4850 [00:00<?, ?it/s]

Dense adversarial avg nDCG@10: 0.6168


In [31]:
# === 12. Fusion sweep on adversarial query set ===

K_RED = 10
K_IRREL = 10
fusion_rows_adv = []

for a in range(0, 101, 5):
    alpha = a / 100.0
    run_fused_adv = fuse_runs_minmax(
        run_dense_adv,
        run_bm25_adv,
        alpha,
        qids=queries_adv_all.keys(),
        k=K_RETRIEVE,
    )

    res = evaluator_adv.evaluate(run_fused_adv)
    avg_ndcg = float(np.mean([r["ndcg_cut_10"] for r in res.values()]))
    avg_red = avg_redundancy_for_run(
        run_fused_adv,
        doc_emb,
        docid_to_i,
        qids=queries_adv_all.keys(),
        k_red=K_RED,
    )
    avg0, rate0, _ = irrelevance_summary(run_fused_adv, qrels_adv_all, queries_adv_all, k=K_IRREL)

    fusion_rows_adv.append({
        "alpha_dense": alpha,
        "beta_bm25": 1.0 - alpha,
        "ndcg@10": avg_ndcg,
        f"redundancy@{K_RED}": avg_red,
        f"irrelevance@{K_IRREL}_avg_count": avg0,
        f"irrelevance@{K_IRREL}_rate": rate0,
    })

fusion_adv_df = pd.DataFrame(fusion_rows_adv)
display(fusion_adv_df)

best_row_adv = fusion_adv_df.loc[fusion_adv_df["ndcg@10"].idxmax()]
print(
    f"Best adversarial fusion by nDCG@10 -> alpha={best_row_adv['alpha_dense']:.2f}, "
    f"beta={best_row_adv['beta_bm25']:.2f}, ndcg@10={best_row_adv['ndcg@10']:.4f}"
)

,alpha_dense,beta_bm25,ndcg@10,redundancy@10,irrelevance@10_avg_count,irrelevance@10_rate
0,0.00,1.00,0.367650,0.464720,5.421443,0.542144
1,0.05,0.95,0.380262,0.473879,5.295464,0.529546
2,0.10,0.90,0.397512,0.485522,5.117320,0.511732
3,0.15,0.85,0.416606,0.498094,4.949485,0.494948
4,0.20,0.80,0.436321,0.511455,4.774433,0.477443
5,0.25,0.75,0.455281,0.525230,4.581443,0.458144
6,0.30,0.70,0.477335,0.541647,4.353608,0.435361
7,0.35,0.65,0.500366,0.558961,4.115052,0.411505
8,0.40,0.60,0.524680,0.578990,3.855876,0.385588
9,0.45,0.55,0.549060,0.600903,3.593402,0.359340


Best adversarial fusion by nDCG@10 -> alpha=0.85, beta=0.15, ndcg@10=0.6207


In [32]:
# === 13. Redundancy and irrelevance diagnostics on adversarial query set ===

avg_redundancy_bm25_adv = avg_redundancy_for_run(
    run_bm25_adv, doc_emb, docid_to_i, qids=queries_adv_all.keys(), k_red=10
)
avg_redundancy_dense_adv = avg_redundancy_for_run(
    run_dense_adv, doc_emb, docid_to_i, qids=queries_adv_all.keys(), k_red=10
)

redundancy_jacc_bm25_adv = {
    qid: avg_pairwise_jaccard(topk_docids_from_run(run_bm25_adv, qid, k=10), doc_token_sets)
    for qid in queries_adv_all.keys()
}
avg_redundancy_jacc_bm25_adv = float(np.mean(list(redundancy_jacc_bm25_adv.values())))

max_redund_dense_adv = {
    qid: max_pairwise_cosine_from_docids(topk_docids_from_run(run_dense_adv, qid, k=10), doc_emb, docid_to_i)
    for qid in queries_adv_all.keys()
}
avg_max_redund_dense_adv = float(np.mean(list(max_redund_dense_adv.values())))

avg0_bm25_adv, rate0_bm25_adv, _ = irrelevance_summary(run_bm25_adv, qrels_adv_all, queries_adv_all, k=10)
avg0_dense_adv, rate0_dense_adv, _ = irrelevance_summary(run_dense_adv, qrels_adv_all, queries_adv_all, k=10)

summary_adv_df = pd.DataFrame([
    {
        "run": "BM25_adv",
        "ndcg@10": avg_ndcg10_bm25_adv,
        "semantic_redundancy@10": avg_redundancy_bm25_adv,
        "lexical_redundancy@10": avg_redundancy_jacc_bm25_adv,
        "irrelevance@10_avg_count": avg0_bm25_adv,
        "irrelevance@10_rate": rate0_bm25_adv,
    },
    {
        "run": "Dense_adv",
        "ndcg@10": avg_ndcg10_dense_adv,
        "semantic_redundancy@10": avg_redundancy_dense_adv,
        "max_pairwise_cosine@10": avg_max_redund_dense_adv,
        "irrelevance@10_avg_count": avg0_dense_adv,
        "irrelevance@10_rate": rate0_dense_adv,
    },
])

display(summary_adv_df)

,run,ndcg@10,semantic_redundancy@10,lexical_redundancy@10,irrelevance@10_avg_count,irrelevance@10_rate,max_pairwise_cosine@10
0,BM25_adv,0.367650,0.464776,0.135833,5.423918,0.542392,NaN
1,Dense_adv,0.616831,0.702822,NaN,2.795052,0.279505,0.953413


In [33]:
# === 14. Compare original vs adversarial overall results ===

compare_overall_df = pd.DataFrame([
    {
        "setting": "Original",
        "bm25_ndcg@10": avg_ndcg10_bm25,
        "dense_ndcg@10": avg_ndcg10_dense,
        "best_fused_ndcg@10": fusion_df["ndcg@10"].max(),
        "best_alpha": fusion_df.loc[fusion_df["ndcg@10"].idxmax(), "alpha_dense"],
        "bm25_irrel@10": avg0_bm25,
        "dense_irrel@10": avg0_dense,
    },
    {
        "setting": "Adversarial",
        "bm25_ndcg@10": avg_ndcg10_bm25_adv,
        "dense_ndcg@10": avg_ndcg10_dense_adv,
        "best_fused_ndcg@10": fusion_adv_df["ndcg@10"].max(),
        "best_alpha": fusion_adv_df.loc[fusion_adv_df["ndcg@10"].idxmax(), "alpha_dense"],
        "bm25_irrel@10": avg0_bm25_adv,
        "dense_irrel@10": avg0_dense_adv,
    },
])

display(compare_overall_df)

,setting,bm25_ndcg@10,dense_ndcg@10,best_fused_ndcg@10,best_alpha,bm25_irrel@10,dense_irrel@10
0,Original,0.461049,0.674993,0.677212,0.90,4.474227,2.226804
1,Adversarial,0.367650,0.616831,0.620665,0.85,5.423918,2.795052
